# **Lab 2: Advanced Visualization with Plotly**

**Course**: **INS-605: Data Analysis II** <br>
**Lecturer**: **Sothea HAS, PhD**

-----

**Objective:** In this lab, we will use the cleaned **Amazon Product Reviews** dataset from Lab 1 to practice interactive visualization with **Plotly**. You will build individual figures first, then combine selected figures into a small dashboard-style view.

The lab focuses on:

- choosing an appropriate chart for a question;
- creating interactive Plotly Express figures;
- controlling titles, labels, hover information, and axes;
- comparing product and customer activity;
- visualizing rating distributions and trends over time;
- assembling multiple figures with `make_subplots()`.

> The notebook of the lab can be downloaded from [Lab2: Advanced Data Visualization](https://hassothea.github.io/AUPP_Data_Analysis_II/Labs/Lab2/02-advanced-visualization.ipynb){target="_blank"}.
> 
-----

- Student name: ...
- ID: ...

-----

## **0. Setup and Prepare the Data**

We use the **same [`Kaggle Amazon Product Reviews dataset`](https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews){target='_blank'} as Lab 1** a. If you already have the cleaned `data` DataFrame from Lab 1, you may reuse it or revise it as we go through the questions below.

For this lab, the important columns are:

`UserId`, `ProductId`, `Score`, `Time`, `ProfileName`, `Summary`, and the engineered fields such as `date`, `year`, and `month`.

In [1]:
#| echo: true
#| code-fold: true

# If data is already available from Lab 1, keep this cell simple.
# Otherwise, run the following code.

# %pip install kagglehub

import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("arhamrumi/amazon-product-reviews")
file_name = [f for f in os.listdir(path) if f.endswith(".csv")][0]
data = pd.read_csv(f"{path}/{file_name}")

data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [2]:
# add columns date, month and year into the data
data['date'] = pd.to_datetime(
    data['Time'],
    unit='s')
data['Month'] = data['date'].dt.to_period('M')
data['Year'] = data['date'].dt.year
data.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,date,Month,Year
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,2011-04-27,2011-04,2011
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,2012-09-07,2012-09,2012
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,2008-08-18,2008-08,2008
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,2011-06-13,2011-06,2011
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,2012-10-21,2012-10,2012


### **0.1. Quick preparation**

Use the cleaning work from Lab 1. At minimum, make sure `Time` is converted to a datetime column named `date`.

**Question A.1.** How many observations and variables are available for visualization?

In [3]:
data.shape

(568454, 13)

In [4]:
data.drop(columns=['Id'], inplace=True) # modify the data orginal data

**A.2.** This data contains more detailed information that you can inspect.

- Drop duplications before and after removing column `ProductId`. What do you observe?
- Group the data by `ProductId` and `UserId`, then compute the count the size of each group and sort them in descending order.
- Inspect the `summary`, `text`, `score` and `time` of those reviews. What do you think?
- Drop duplicated reviews for each product and keep only those with largest number of `HelpfulnessDenominator`.

In [5]:
n = data.shape[0]
n1 = data.drop_duplicates().shape[0]
n2 = data.drop(columns=['ProductId'])\
        .drop_duplicates()\
            .shape[0]
print(f'* The number of observations after dropping the dupplications: {n1}. {round((n-n1)/n * 100, 2)}% is gone.')
print(f'* The number of observations after dropping column `ProductId` and then the duplications: {n2}. {round((n-n2)/n * 100, 2)}% is gone.')

* The number of observations after dropping the dupplications: 568173. 0.05% is gone.
* The number of observations after dropping column `ProductId` and then the duplications: 396309. 30.28% is gone.


In [6]:
data.groupby(
    ['ProductId', 'UserId', 'Text'])\
    .agg(
        Count = ('Score', 'size'),
        Largest = ('HelpfulnessDenominator', 'max')
    )\
    .sort_values('Count', ascending=False)\
    .reset_index()\
    .drop(columns=['Text'])

,ProductId,UserId,Count,Largest
0,B003M60K54,A3TVZM3ZIXG8YW,10,48
1,B000084DWM,A3TVZM3ZIXG8YW,10,48
2,B0006345PW,A3TVZM3ZIXG8YW,10,48
3,B009B87SAC,A3TVZM3ZIXG8YW,10,48
4,B000QSN7P6,A3TVZM3ZIXG8YW,10,48
...,...,...,...,...
567140,B00113ZZ5U,A1MOH4G96RL54O,1,61
567141,B00113ZZ5U,A1JR3GSIEDZO3S,1,6
567142,B00113ZZ5U,A1BT3TD2FVRK0K,1,9
567143,B00113ZZ5U,A12PHJ1DNFIA6,1,14


In [7]:
data.query("UserId == 'A29JUMRL1US6YP' and ProductId in ['B000WFKWDI', 'B000WFKI82', 'B000WFORH0']")[['ProductId', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'UserId', 'date', 'Score', 'Text', 'Summary']]

,ProductId,HelpfulnessNumerator,HelpfulnessDenominator,UserId,date,Score,Text,Summary
146434,B000WFKWDI,3,4,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146477,B000WFKWDI,38,40,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146483,B000WFKWDI,19,23,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146486,B000WFKWDI,12,14,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146493,B000WFKWDI,7,8,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146497,B000WFKWDI,4,4,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146510,B000WFKWDI,3,3,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146511,B000WFKWDI,3,3,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146518,B000WFKWDI,5,6,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health
146543,B000WFKWDI,2,2,A29JUMRL1US6YP,2010-07-04,5,The pet food industry can be one of the most i...,Fantastic Food for Good Cat Health


In [8]:
data.query("UserId == 'AF3BYMPWKWO8F' and ProductId == 'B001BCXTGS'")[['ProductId', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'UserId', 'date', 'Score', 'Text', 'Summary']]

,ProductId,HelpfulnessNumerator,HelpfulnessDenominator,UserId,date,Score,Text,Summary
139600,B001BCXTGS,0,0,AF3BYMPWKWO8F,2008-12-18,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139675,B001BCXTGS,91,98,AF3BYMPWKWO8F,2008-11-25,3,While several reviewers have alluded to the he...,Please check the controversial ingredients and...
139677,B001BCXTGS,20,23,AF3BYMPWKWO8F,2009-05-29,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139679,B001BCXTGS,6,6,AF3BYMPWKWO8F,2008-12-18,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139682,B001BCXTGS,7,8,AF3BYMPWKWO8F,2009-05-29,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139708,B001BCXTGS,0,1,AF3BYMPWKWO8F,2009-05-29,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139743,B001BCXTGS,6,7,AF3BYMPWKWO8F,2008-12-18,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139747,B001BCXTGS,3,3,AF3BYMPWKWO8F,2009-05-29,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione
139748,B001BCXTGS,3,3,AF3BYMPWKWO8F,2009-05-29,1,"According to the manufacturer's website, this ...",Warning: Contains Menadione


In [9]:
idx = data.groupby(
    ['UserId', 'Text']
    )['HelpfulnessDenominator'].idxmax()

print(f"Retained index size: {idx.shape}")
print(f'Percentage of retained rows: {idx.shape[0]/data.shape[0]}')

Retained index size: (393606,)
Percentage of retained rows: 0.6924148655827912


In [10]:
data.loc[idx].shape

(393606, 12)

-------

## **1. First Interactive Plot: Rating Distribution**

A dashboard often starts with a simple overview of the target variable.

**Question B.** What is the distribution of review ratings (`Score`)?

Create a chart showing the **number of reviews for each rating**.

Requirements:

- x-axis: rating;
- y-axis: number of reviews;
- display the count on the bars;
- add a meaningful title;
- include a useful hover tooltip.

**Hint:** `px.bar()`, `.value_counts()`, `.reset_index()`, `text=`, `hover_data=`.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

score_table = data['Score'].value_counts()

fig_score = go.Figure()

fig_score.add_trace(
    go.Bar(
        x=score_table.index,
        y=score_table.values,
        text=score_table.values,
        hovertemplate="Score count: %{y:}"
    )
)
fig_score.update_layout(
    title='Bar chart of score',
    width=800,
    height=500
)
fig_score.show()

### **1.1. Improve the interaction**

A useful interactive chart should help the viewer understand the data without reading the code.

**Question C.** Modify your figure so that:

1. the x-axis is labeled **Rating**;
2. the y-axis is labeled **Number of Reviews**;
3. the bar labels show the review counts;
4. the hover tooltip shows both rating and review count.

**Hint:** `fig.update_layout()`, `fig.update_traces()`, `texttemplate`, `hovertemplate`.

In [12]:
# To do

-------

## **2. Product Comparison**

Product managers may want to know which products receive the most reviews.

First calculate the **top 10 products by number of reviews**.

**Question D.**

- Which 10 products receive the most reviews?
- Create a horizontal bar chart.
- Sort the products from highest to lowest review count.

**Hint:** `groupby()`, `.size()`, `.sort_values()`, `.head()`, `px.bar(..., orientation="h")`.

:::{.callout-tip}
**Visualization hint:** Product IDs are categorical labels. A horizontal bar chart is usually easier to read than a vertical chart when labels are long and there is no size constaint. Otherwise, vertical bar chart can also be use with rotated ticks (`tickanlge = ...`).
:::

In [13]:
# To do

### **2.1. Add another dimension**

The number of reviews alone does not tell us whether customers are satisfied.

For the top 10 reviewed products, calculate:
- number of reviews;
- average rating.

**Question E.** Build a scatter plot where:
- x = number of reviews;
- y = average rating;
- each point represents a product;
- point size represents review volume;
- hover information includes the product ID.

**Hint:** `groupby().agg()`, `px.scatter()`, `size=`, `hover_name=`.

In [14]:
# To do

-------

## **3. Customer View**

A dashboard may also focus on customer activity.

**Question F.** Find the **top 15 customers by number of reviews** and create a horizontal bar chart.

Then ask:

> Are the most active customers necessarily the most satisfied?

**Hint:** `groupby("UserId").agg(...)`, `sort_values()`, `px.bar()`.

In [15]:
# To do

### **3.1. Compare activity and satisfaction**

Create a customer-level summary containing:
- number of reviews;
- average rating.

**Question G.** Create a scatter plot with:
- x = number of reviews;
- y = average rating;
- one point per customer.

Use a logarithmic x-axis if the number of reviews is highly skewed.

**Hint:** `px.scatter()`, `fig.update_xaxes(type="log")`.

:::{.callout-note}
Do not assume that a high number of reviews means a customer is highly satisfied. Let the visualization show the relationship.
:::

In [16]:
# To do

-------

## **4. Rating Trend Over Time**

A dashboard should help identify changes in customer behavior.

Convert `Time` to a monthly date and calculate:
- number of reviews per month;
- average rating per month.

**Question H.** Create a line chart of monthly review volume.

Then create a second line showing the monthly average rating.

**Hint:** `pd.to_datetime()`, `.dt.to_period("M")`, `groupby()`, `px.line()`.

In [17]:
# To do

### **4.1. Make the trend easier to interpret**

Add:
- markers to the line;
- a clear title;
- axis labels;
- hover information.

**Question I.** Are there periods with unusually high or low review activity?

**Hint:** `markers=True`, `hover_data=`, `update_layout()`.

In [18]:
# To do

-------

## **5. Rating Composition Over Time**

A single average rating can hide important changes in the rating composition.

Create a monthly table containing the number of **1-, 2-, 3-, 4-, and 5-star reviews**.

**Question J.** Create a **100% stacked area chart** showing how the rating composition changes over time.

The five ratings at each month should sum to approximately **100%**.

**Hint:**
- `pd.crosstab()`;
- `div(..., axis=0)`;
- `px.area()`;
- `groupnorm="percent"` can be useful when using a long-form table.

:::{.callout-warning}
Do not plot the raw counts and call them percentages. Convert the monthly counts to proportions or percentages first.
:::

In [19]:
# To do

-------

## **6. Assemble a Small Dashboard**

You have now built several individual views. A dashboard combines related views so that a user can answer several questions at once.

Build a compact **Overview Dashboard** using `make_subplots()` with:

- **KPI 1:** Total reviews
- **KPI 2:** Unique customers
- **KPI 3:** Unique products
- **KPI 4:** Average rating
- **Chart 1:** Rating distribution
- **Chart 2:** Top 10 products
- **Chart 3:** Monthly review activity

**Question K.** What information should a manager understand from the dashboard in less than 10 seconds?

**Hint:** `go.Indicator()`, `go.Bar()`, `go.Scatter()`, `make_subplots()`, `add_trace()`, `update_layout()`.

In [20]:
# To do

### **6.1. Final dashboard improvements**

Improve your dashboard with at least **three** of the following:

- [ ] meaningful chart titles;
- [ ] consistent font;
- [ ] consistent colors;
- [ ] readable axis labels;
- [ ] hover information;
- [ ] number formatting;
- [ ] appropriate margins and spacing;
- [ ] a single overall dashboard title;
- [ ] remove unnecessary legends or gridlines.

**Question L.** Which design change makes your dashboard easier to read, and why?

In [21]:
# To do

-------

## **7. Reflection**

Answer briefly:

1. Which visualization was most useful for understanding the dataset?
2. Which chart would you show to a **product manager**?
3. Which chart would you show to a **marketing manager**?
4. What is one limitation of your dashboard?

:::{.callout-tip}
The goal is not to make the dashboard as complicated as possible. A good dashboard communicates a small number of important ideas clearly.
:::